# Workshop: Security & Governance

Practice security & governance with hands-on exercises.

| Duration | Format | Difficulty |
|---|---|---|
| 30 min | Hands-on Workshop | Intermediate |

**Prerequisites:** 04 — Security & Governance Demo

<!-- TRAINER-BOX -->

> **TRAINER INSTRUCTIONS**
>
> | Tier | Target audience | Time | Goal |
> |---|---|---|---|
> | **PART 1 — FUNDAMENTAL** | New to Spark / Databricks | 60 min | Building confidence, fill-in-blank |
> | **PART 2 — ADVANCED** | 1+ year experience | 65 min | Debugging, optimization, design from scratch |
>
> - **Beginner** groups: complete all PART 1, PART 2 optional
> - **Experienced** groups: PART 1 as quick review (15 min), focus on PART 2
> - **Mixed** groups: PART 1 mandatory, PART 2 for faster participants

<!-- PART1-FUNDAMENTAL -->

# PART 1 — FUNDAMENTAL (Fundamental Level)

<!-- LAB-SCENARIO -->

## Scenario

> *"The organization stores customer PII in Unity Catalog. Your task is to protect sensitive columns with masking functions, restrict row visibility with row-level security filters, and verify the applied controls using audit metadata."*

<!-- LAB-OBJECTIVES -->

## Learning Objectives

After completing this lab you will be able to:

- Create and apply **column masking functions** to protect PII (email)
- Implement **row-level security** (RLS) using row filter functions
- Apply and remove masking / row filter controls on Delta tables
- Audit applied controls with `INFORMATION_SCHEMA.TABLE_PRIVILEGES`

## Setup

In [ ]:
%run ../../setup/00_setup

In [ ]:
# Recreate Bronze tables from dataset files (in case they don't exist)
for tbl, fmt, path in [
    ("customers", "csv", f"{DATASET_PATH}/customers/customers.csv"),
    ("orders", "json", f"{DATASET_PATH}/orders/orders_batch.json"),
    ("products", "csv", f"{DATASET_PATH}/products/products.csv"),
]:
    full_name = f"{CATALOG}.{BRONZE_SCHEMA}.{tbl}"
    if not spark.catalog.tableExists(full_name):
        reader = spark.read.format(fmt)
        if fmt == "csv":
            reader = reader.option("header", "true").option("inferSchema", "true")
        reader.load(path).write.mode("overwrite").saveAsTable(full_name)
        print(f"[RECREATED] {full_name}")
    else:
        print(f"[OK] {full_name} exists")

In [ ]:
# Ensure Silver tables exist (create from Bronze if missing)
silver_customers = f"{CATALOG}.{SILVER_SCHEMA}.customers"
silver_orders = f"{CATALOG}.{SILVER_SCHEMA}.orders"

if not spark.catalog.tableExists(silver_customers):
    spark.sql(f"""
        CREATE TABLE {silver_customers} AS
        SELECT * FROM {CATALOG}.{BRONZE_SCHEMA}.customers
    """)
    print(f"Created {silver_customers} from Bronze")

if not spark.catalog.tableExists(silver_orders):
    spark.sql(f"""
        CREATE TABLE {silver_orders} AS
        SELECT *,
            CASE 
                WHEN CAST(REGEXP_EXTRACT(store_id, '(\\\\d+)') AS INT) <= 10 THEN 'East'
                WHEN CAST(REGEXP_EXTRACT(store_id, '(\\\\d+)') AS INT) <= 20 THEN 'West'
                ELSE 'Central'
            END AS store_region
        FROM {CATALOG}.{BRONZE_SCHEMA}.orders
    """)
    print(f"Created {silver_orders} from Bronze (with store_region)")

df_customers = spark.table(silver_customers)
df_orders = spark.table(silver_orders)
print(f"Customers: {df_customers.count()}, Orders: {df_orders.count()}")

## Task 1: Create a Column Mask Function ~4 min

Create a SQL function that masks email addresses. Non-admin users see only the first 2 characters.

**What you need to do:** Fill in the blanks:
1. `________(________)` → `is_account_group_member('admins')`

**Guidance — Task 01**

The goal is to create a **column masking function** — a SQL UDF that dynamically hides sensitive data based on who is querying.

**How column masking works**
A masking function takes the original column value and returns either the real value or a masked version. The decision is based on `is_account_group_member('group_name')` — a built-in function that returns TRUE if the current user belongs to the specified group. Admins see real data; everyone else sees masked data.

**Function signature**
The function must accept the same type as the column it will mask (STRING for email) and return the same type. Use `CASE WHEN ... THEN ... ELSE ...` for the branching logic.

**Things to think about**
- What masking strategy is appropriate — full redaction, partial (first 2 chars), or tokenization?
- Can a user bypass masking by creating a view on top of the masked table?

In [ ]:
# TODO: Create masking function
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.mask_email(email STRING)
    RETURNS STRING
    RETURN CASE
        WHEN ________(________) THEN email
        ELSE CONCAT(LEFT(email, 2), '***@***.***')
    END
""")
print("Masking function created")

In [ ]:
# Verification -- test the function
test_df = spark.sql(f"SELECT {CATALOG}.{SILVER_SCHEMA}.mask_email('john.doe@example.com') AS masked")
test_df.display()
print("Task 1 PASSED")

## Task 2: Apply Column Mask to Table ~3 min

Apply the `mask_email` function to the `email` column of the `customers` table.

**What you need to do:** Fill in the blanks:

1. `SET ________` → `MASK`2. Function name → `mask_email`

**Guidance — Task 02**

The goal is to **bind the masking function to a table column** — making masking automatic for every query.

**ALTER TABLE ... SET MASK**
The syntax `ALTER TABLE t ALTER COLUMN col SET MASK function_name` associates the masking function with a specific column. From this point on, every `SELECT` on that column runs through the masking function — no application code changes needed.

**Transparent enforcement**
This is a key advantage over application-level masking. Even ad-hoc SQL, BI tools, and notebooks see masked data. The masking is enforced by Unity Catalog at the query engine level — it cannot be bypassed by the end user.

**Things to think about**
- What happens to existing views that reference the masked column — do they see masked data too?
- Can you apply different masking functions to different columns in the same table?

In [ ]:
# TODO: Apply column mask
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.customers
    ALTER COLUMN email SET ________ {CATALOG}.{SILVER_SCHEMA}.________
""")
print("Column mask applied to email column")

In [ ]:
# Verification -- query the table
masked_df = spark.sql(f"SELECT customer_id, email FROM {CATALOG}.{SILVER_SCHEMA}.customers LIMIT 5")
masked_df.display()
print("Task 2 PASSED -- check if email is masked for non-admin users")

## Task 3: Create a Row Filter Function ~5 min

Create a function that restricts visibility of orders by `store_region`. Only users in the matching group see rows for their region. Admins see all.

**What you need to do:** Fill in group names and region values (e.g., `'east_team'` / `'East'`, `'west_team'` / `'West'`)

**Guidance — Task 03**

The goal is to create a **row filter function** — Row-Level Security (RLS) that limits which rows a user can see.

**How RLS works in Unity Catalog**
A row filter function takes one or more column values as input and returns a BOOLEAN. When applied to a table, Unity Catalog adds a `WHERE` clause to every query: only rows where the function returns TRUE are visible. This happens transparently — the user doesn't know rows are being filtered.

**Design pattern: group-based access**
Use `is_account_group_member('group')` to check the user's group membership, then match against the row's data. Admins typically bypass all filters (return TRUE unconditionally). Regional teams see only their region's data.

**Things to think about**
- How does RLS affect `COUNT(*)` — does it count all rows or only visible rows?
- Can a user determine that rows are being filtered from them?

In [ ]:
# TODO: Create row filter function
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {CATALOG}.{SILVER_SCHEMA}.region_filter(region STRING)
    RETURNS BOOLEAN
    RETURN (
        is_account_group_member('admins')
        OR (is_account_group_member('________') AND region = '________')
        OR (is_account_group_member('________') AND region = '________')
    )
""")
print("Row filter function created")

## Task 4: Apply Row Filter to Table ~3 min

Apply the `region_filter` function to the `orders` table.

**What you need to do:** Fill in the blanks:

1. `SET ________` → `ROW FILTER`2. `ON (________)` → `store_region`

**Guidance — Task 04**

The goal is to **bind the row filter to the table** — activating RLS.

**ALTER TABLE ... SET ROW FILTER**
The syntax is `ALTER TABLE t SET ROW FILTER function_name ON (column)`. The `ON (column)` specifies which column(s) are passed as arguments to the filter function. If the function takes `region STRING`, then `ON (store_region)` passes the `store_region` column value for each row.

**Immediate effect**
Once set, the filter applies to all subsequent queries — including running notebooks, BI dashboards, and SQL queries. Admins (who pass the `is_account_group_member('admins')` check) still see all rows.

**Things to think about**
- Can you combine RLS and column masking on the same table?
- What happens to streaming queries that read from a table with RLS?

In [ ]:
# TODO: Apply row filter
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.orders
    SET ________ {CATALOG}.{SILVER_SCHEMA}.region_filter ON (________)
""")
print("Row filter applied to orders table")

In [ ]:
# Verification -- check row count (admins should see all rows)
filtered_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.{SILVER_SCHEMA}.orders").first()["cnt"]
print(f"Visible rows: {filtered_count}")
print("Task 4 PASSED")

## Task 5: Query Table Privileges ~3 min

Use `INFORMATION_SCHEMA.TABLE_PRIVILEGES` to verify who has access to what.

**What you need to do:** Fill in `________` → `TABLE_PRIVILEGES`

**Guidance — Task 05**

The goal is to **audit permissions** — verifying who has access to what using system metadata.

**TABLE_PRIVILEGES view**
`INFORMATION_SCHEMA.TABLE_PRIVILEGES` shows every GRANT at the table level: who granted it (`grantor`), who received it (`grantee`), and what privilege (`SELECT`, `MODIFY`, etc.). This is your auditing tool — use it before security reviews, compliance checks, or troubleshooting access issues.

**Regular auditing**
In production, you should periodically query privileges to detect drift — permissions granted directly to users instead of groups, overly broad grants, or stale grants for departed team members.

**Things to think about**
- How would you find tables that have NO grants (potentially orphaned)?
- Can you see grants from other catalogs, or only the current one?

In [ ]:
# TODO: Query table privileges
privs_df = spark.sql(f"""
    SELECT grantor, grantee, table_schema, table_name, privilege_type
    FROM {CATALOG}.INFORMATION_SCHEMA.________
    ORDER BY grantee, table_name
""")
privs_df.display()

In [ ]:
# Verification
assert privs_df.count() > 0, "No privileges found"
print(f"Found {privs_df.count()} privilege entries")
print("Task 5 PASSED")

## Task 6: Remove Mask and Filter (Cleanup) ~2 min

Remove the column mask and row filter applied earlier.

**What you need to do:** Fill in the blanks:

1. `________ ________` → `DROP MASK`2. `________ ________ ________` → `DROP ROW FILTER`

**Guidance — Task 06**

The goal is to **clean up security controls** — removing column masks and row filters when they're no longer needed.

**DROP MASK and DROP ROW FILTER**
`ALTER TABLE t ALTER COLUMN col DROP MASK` removes the masking function binding from a column. `ALTER TABLE t DROP ROW FILTER` removes the RLS binding. The functions themselves still exist — you're just unbinding them from the table.

**Why cleanup matters**
In a training environment, leftover masks and filters can confuse subsequent labs. In production, you'd typically only remove these during migrations or when replacing with updated versions.

**Things to think about**
- If you `DROP MASK` and then re-run `DESCRIBE TABLE`, does the mask column show anything?
- What happens if you drop the function itself while it's still bound to a table?

In [ ]:
# TODO: Remove column mask from email
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.customers
    ALTER COLUMN email ________ ________
""")

# TODO: Remove row filter from orders
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SILVER_SCHEMA}.orders
    ________ ________ ________
""")

print("Cleanup complete -- mask and filter removed")

In [ ]:
# Final verification
clean_df = spark.sql(f"SELECT customer_id, email FROM {CATALOG}.{SILVER_SCHEMA}.customers LIMIT 3")
clean_df.display()
print("Task 6 PASSED -- all governance controls removed")

<!-- PART2-ADVANCED -->

# PART 2 — ADVANCED (Bonus — If Time Permits)

> **For whom:** Participants with 1+ year of Spark/Databricks experience
>
> **Rules:**
> - No scaffold `= None` — write from scratch
> - Tasks described as JIRA tickets (requirements + acceptance criteria)
> - Complete **at least 2 of 4 challenges**
> - Time per challenge: 8-15 minutes

### Edge Case — Column Masking for Service Account

Column masking does not work for a service account used by a production pipeline.

```sql
CREATE FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN CASE WHEN is_account_group_member('retail_engineers')
            THEN email
            ELSE regexp_replace(email, '(.).*(@.*)', '$1***$2')
       END;
```

The service principal is NOT in the `retail_engineers` group, but still sees full emails. Why?

**Acceptance criteria:**
- Explanation: `is_account_group_member()` vs SP identity
- Fix: add SP to group OR use `current_user()` with an allowlist
- Demonstration of the fixed function with a test call
- Alternative: ABAC with custom attribute on SP

In [ ]:
# YOUR SOLUTION — Column Masking for Service Account
# ------------------------------------------------------------



### Multi-Step Pipeline — RLS + Column Masking — Combination

Build complete security for the `silver_customers` table from scratch:

**Simultaneous requirements:**
1. **RLS**: regional analysts see ONLY their region (`WHERE region = current_user_region()`)
2. **Column Masking**: `email` and `phone` masked for non-engineers
3. **Test**: log in as different users and check what they see

**Acceptance criteria:**
- RLS and masking functions both active on the same table
- `SELECT` as analyst_eu: EU region only + masked PII
- `SELECT` as data_engineer: entire world + full PII
- `DESCRIBE EXTENDED silver_customers` shows both functions

In [ ]:
# YOUR SOLUTION — RLS + Column Masking — Combination
# ------------------------------------------------------------



### Refactor — DBFS Mount → Unity Catalog Volume

A legacy pipeline uses an old DBFS mount. Migrate to UC Volume.

```python
# Old code:
dbutils.fs.mount(
    source='wasbs://landing@retailstorage.blob.core.windows.net/',
    mount_point='/mnt/landing'
)
df = spark.read.csv('/mnt/landing/customers.csv')
spark.sql("GRANT READ_FILES ON ANY FILE TO analysts") # old syntax
```

**Acceptance criteria:**
- Old mount replaced by External Volume
- `spark.read.csv('/Volumes/...')` works
- UC `GRANT READ VOLUME ON VOLUME v TO analysts` — correct syntax
- Written list of migration steps (can be reused as a runbook)

In [ ]:
# YOUR SOLUTION — DBFS Mount → Unity Catalog Volume
# ------------------------------------------------------------



## Part 2 Summary

You have completed the Advanced section for **Security & Governance**.

**Reflection (optional):**
- Which challenge was the hardest and why?
- What would you change in your implementation?
- What approach would you use in production?

## Summary

| Task | Concept | Key SQL |
|------|---------|--------|
| 1-2 | Column Mask | `ALTER TABLE ... ALTER COLUMN ... SET MASK fn` |
| 3-4 | Row Filter | `ALTER TABLE ... SET ROW FILTER fn ON (col)` |
| 5 | Audit | `INFORMATION_SCHEMA.TABLE_PRIVILEGES` |
| 6 | Cleanup | `DROP MASK`, `DROP ROW FILTER` |

← [04 — Security & Governance](../Demo/04_security_governance_demo.ipynb) | **[ README](../../../README.md)**